In [ ]:
%pip -q install transformers==4.51.3 bitsandbytes==0.50.2 accelerate==1.6.0

In [ ]:
PROMPT_DATA = '...'
import base64
import gzip
import hashlib
import json
from pathlib import Path
import time
import re

import bitsandbytes
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Select a GPU under Runtime → Change runtime type."
model_id = "Qwen/Qwen2.5-7B-Instruct"
revision = "a09a35458c702b33eeacc393d103063234e8bc28"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=revision, device_map={"": 0}, torch_dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
).eval()
raw = gzip.decompress(base64.b64decode(PROMPT_DATA))
rows = [json.loads(line) for line in raw.splitlines()]
assert len(rows) == 960

def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def update_memory(memory, block, budget):
    # Preserve complete proposal/reply exchanges. Prioritize naming language.
    pairs = memory + list(zip(block[::2], block[1::2]))
    keep = list(range(len(pairs)))
    priority = lambda i: (bool(re.search(
        r"\b(?:label|name|naming|nickname|shorthand|refer|mapping|call|proposal|adopt)\b",
        pairs[i][0].split(": ", 1)[1], re.I
    )), i)
    def render():
        return "\n".join(turn for i in keep for turn in pairs[i])
    while token_count(render()) > budget:
        keep.remove(min(keep, key=priority))
    return [pairs[i] for i in keep]

memories = {}
for row in rows:
    instructions, remainder = row["prompt"].split("\n\nDIALOGUE:\n", 1)
    transcript, _ = remainder.split("\n\nCANDIDATE ENTITIES:\n", 1)
    if row["id"] in memories:
        continue
    turns = transcript.splitlines()
    assert len(turns) == 24
    memories[row["id"]] = {}
    for budget in (128, 256):
        memory = []
        for i in range(0, len(turns), 6):
            memory = update_memory(memory, turns[i:i+6], budget)
        text = "\n".join(turn for pair in memory for turn in pair)
        assert 0 < token_count(text) <= budget
        memories[row["id"]][budget] = text
assert len(memories) == 320

evaluation = []
for row in rows:
    original, tail = row["prompt"].split("\n\nCANDIDATE ENTITIES:\n", 1)
    instructions = original.split("\n\nDIALOGUE:\n", 1)[0]
    instructions = instructions.replace("Read the dialogue", "Read the dialogue memory")
    for budget, memory in memories[row["id"]].items():
        prompt = instructions + "\n\nDIALOGUE MEMORY:\n" + memory
        prompt += "\n\nCANDIDATE ENTITIES:\n" + tail
        evaluation.append(dict(row, prompt=prompt, budget=budget, memory_tokens=token_count(memory)))
rows = evaluation
prepared = [(row, tokenizer.apply_chat_template(
    [{"role": "user", "content": row["prompt"]}], tokenize=False, add_generation_prompt=True,
)) for row in rows]
prepared.sort(key=lambda item: len(item[1]))
batch_size = 8
max_new_tokens = 24
print("GPU:", torch.cuda.get_device_name(), flush=True)
start = time.monotonic()
output = Path("qwen_compressed_predictions.jsonl")
with output.open("w", buffering=1) as file, torch.inference_mode():
    for offset in range(0, len(prepared), batch_size):
        batch = prepared[offset:offset + batch_size]
        inputs = tokenizer([prompt for _, prompt in batch], padding=True,
                           add_special_tokens=False, return_tensors="pt").to("cuda")
        assert inputs.input_ids.size(1) + max_new_tokens < model.config.max_position_embeddings
        generated = model.generate(
            **inputs, do_sample=False, max_new_tokens=max_new_tokens,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )
        continuations = generated[:, inputs.input_ids.size(1):]
        for (row, _), answer, length in zip(
            batch, tokenizer.batch_decode(continuations, skip_special_tokens=True),
            inputs.attention_mask.sum(1).tolist(),
        ):
            file.write(json.dumps({
                "id": row["id"], "question_type": row["question_type"],
                "prediction": answer.strip(), "prompt_tokens": length,
                "method": f"qwen_extractive_{row['budget']}",
                "memory_tokens": row["memory_tokens"],
            }) + "\n")
        if offset % 64 == 0 or offset + len(batch) == len(prepared):
            print(f"{offset + len(batch)}/{len(prepared)} ({time.monotonic() - start:.0f}s)", flush=True)
metadata = {
    "model": model_id, "revision": revision, "quantization": "bitsandbytes NF4, float16 compute",
    "gpu": torch.cuda.get_device_name(),
    "memory_method": "recursive extractive, naming cues first, recency breaks ties",
    "memory_budgets": [128, 256], "transformers": transformers.__version__,
    "torch": torch.__version__, "bitsandbytes": bitsandbytes.__version__,
    "batch_size": batch_size, "max_new_tokens": max_new_tokens, "do_sample": False,
    "prompt_file_sha256": hashlib.sha256(raw).hexdigest(),
}
print(json.dumps(metadata, indent=2))
Path("qwen_compressed_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")


In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile("qwen_compressed_results.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write("qwen_compressed_predictions.jsonl")
    archive.write("qwen_compressed_metadata.json")
files.download("qwen_compressed_results.zip")